In [12]:
import re

In [20]:
def parse_log_file(file_path):
    try:
        received_data = {}
        sig_pattern = re.compile(r'\[(\d+)/1024\] Received sig\s+\.\.\.\s+([0-9a-fA-F\s]+)')
        root_pattern = re.compile(r'\[(\d+)/1024\] Received root\s+\.\.\.\s+([0-9a-fA-F]+)')
        sig_dict = {}
        
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                sig_match = sig_pattern.search(line)
                if sig_match:
                    index = int(sig_match.group(1))
                    sig_data = sig_match.group(2).strip()
                    sig_dict[index] = sig_data
                    if index not in received_data:
                        received_data[index] = {}
                        received_data[index]['sig'] = sig_data
                    continue
                
                root_match = root_pattern.search(line)
                if root_match:
                    index = int(root_match.group(1))
                    root_data = root_match.group(2).strip()
                    if index in sig_dict and 'root' not in received_data[index]:
                        received_data[index]['root'] = root_data
        
        return received_data
        
    except Exception as e:
        print(f"ERROR: {e}")
        return None

In [14]:
def parse_log_file(file_path):
    received_data = {}
    pattern = re.compile(r'\[(\d+)/1024\] Received\s+\.\.\.\s+([0-9a-fA-F\s]+)')
    
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                index = int(match.group(1))
                if index not in received_data:
                    data = match.group(2).strip()
                    received_data[index] = data
                
    return received_data

In [15]:
def split_into_blocks(data):
    return data.split()


In [16]:
def find_differences(received_data, reference_index=1):
    reference_data = received_data.get(reference_index, None)
    print(reference_data)
    print(f"reference_index {reference_index} reference_data: {reference_data}")
    if not reference_data:
        print(f"can't find {reference_index} reference_data")
        return {}
    
    differences = {}
    for index, data in received_data.items():
        if index == reference_index:
            continue

        ref_blocks = split_into_blocks(reference_data['sig'])
        data_blocks = split_into_blocks(data['sig'])
        
        diff_blocks = []
        for i, (ref_block, data_block) in enumerate(zip(ref_blocks, data_blocks), start=1):
            if ref_block != data_block:
                diff_blocks.append(i)
        
        if diff_blocks:
            differences[index] = diff_blocks
    
    return differences

In [17]:
def find_differences_root(received_data, reference_index=1):
    reference_data = received_data.get(reference_index, None)
    print(f"reference_index {reference_index} reference_data: {reference_data}")
    if not reference_data:
        print(f"can't find {reference_index} reference_data")
        return {}
    
    differences = {}
    for index, data in received_data.items():
        if index == reference_index:
            continue
        if data['root'] != reference_data['root']:
            differences[index] = data['root']
    
    return differences

In [18]:
def annotate_differences(data, diff_blocks):
    blocks = split_into_blocks(data['sig'])
    annotated_blocks = []
    for i, block in enumerate(blocks, start=1):
        if i in diff_blocks:
            annotated_blocks.append(f"*{block}*")
        else:
            annotated_blocks.append(block)
    return ' '.join(annotated_blocks)

In [21]:
log_file = '2025-06-20_09-09-01_SPHINCSplus_exp1.txt'


received_data = parse_log_file(log_file)
print("received_data", received_data)

differences = find_differences(received_data, reference_index=1)


print('sig fault:\t', len(differences))

received_data {1: {'sig': 'f53398f8299d24c438553b753633399c 990ad968f185efa2c4f27dfb0a87965c d7d3a39ca3fc5b87bc02743b9da55b39 651a1903e13f2709980b2dcf65bd6471 ea7d7939c569b04c80bdb414d4155394 018df1f72786787ee93898bee6899963 f87584ee6edc1a2e8bbfaea284b69d53 4c3f97f68691da6da8e075e715221bbc 5cfd19dc6e5e99e96b1f48fbdf223279 b956e0798e56bb077a236a7fd8ea6560 d733c1fc07489268739e52e41d3e9fad d6677a1e589f0719060b8731352bc938 1e305851f27f24b83cdf77f5a986b6a8 f9db1048a1110b2d96700959b88040c0 829744a3296445967fa1f29fd16970d1 783f94e4bd7a20d667d22455b948555e 03188eb783c87b9338f8c940fafde27c f36f8e4b108f35e5151efc0da686ef77 123825074c11a42a6595bc386b1891a1 d351d8f2993c302a330f9100b5b53347 2a3ab136773dd3688ba6c6d2fd206ea3 829dad826d5aff767918d81df1a85e65 67803c17c0e2bbb26f5d70ecfdd8e78d 96537d6df2d9420fc37137cb1e2379cf b162fb37eb8a6f7f587441aa117af37d 248c0364fa92a1538cf8102fd55669dc 21efbe4a50acfc92a594523a43942bc7 3c55e82fd314fafb2ec4f5a26bd46aed 979fe6e61e65885d7e6faec0ed855d36 6efe167f37d1d00f

In [22]:
differences_root = find_differences_root(received_data, reference_index=1)

print('root fault:\t', len(differences_root))

reference_index 1 reference_data: {'sig': 'f53398f8299d24c438553b753633399c 990ad968f185efa2c4f27dfb0a87965c d7d3a39ca3fc5b87bc02743b9da55b39 651a1903e13f2709980b2dcf65bd6471 ea7d7939c569b04c80bdb414d4155394 018df1f72786787ee93898bee6899963 f87584ee6edc1a2e8bbfaea284b69d53 4c3f97f68691da6da8e075e715221bbc 5cfd19dc6e5e99e96b1f48fbdf223279 b956e0798e56bb077a236a7fd8ea6560 d733c1fc07489268739e52e41d3e9fad d6677a1e589f0719060b8731352bc938 1e305851f27f24b83cdf77f5a986b6a8 f9db1048a1110b2d96700959b88040c0 829744a3296445967fa1f29fd16970d1 783f94e4bd7a20d667d22455b948555e 03188eb783c87b9338f8c940fafde27c f36f8e4b108f35e5151efc0da686ef77 123825074c11a42a6595bc386b1891a1 d351d8f2993c302a330f9100b5b53347 2a3ab136773dd3688ba6c6d2fd206ea3 829dad826d5aff767918d81df1a85e65 67803c17c0e2bbb26f5d70ecfdd8e78d 96537d6df2d9420fc37137cb1e2379cf b162fb37eb8a6f7f587441aa117af37d 248c0364fa92a1538cf8102fd55669dc 21efbe4a50acfc92a594523a43942bc7 3c55e82fd314fafb2ec4f5a26bd46aed 979fe6e61e65885d7e6faec0ed855d36 

In [23]:
sig_index = list(differences.keys())
root_index = list(differences_root.keys())

In [24]:
diff_sig = set(sig_index) - set(root_index)
print('sig fault but root correct:')
print(diff_sig)
print(len(diff_sig))

sig fault but root correct:
{996, 938, 941, 973, 1007, 912, 913, 983}
8


In [26]:
for index in diff_sig:
    diff_blocks = differences[index]

    # annotated_data = annotate_differences(received_data[index], diff_blocks)
    print(f"index: {index:04d}\t data: {diff_blocks}")


index: 0996	 data: [31]
index: 0938	 data: [13]
index: 0941	 data: [14]
index: 0973	 data: [24]
index: 1007	 data: [35]
index: 0912	 data: [5]
index: 0913	 data: [5]
index: 0983	 data: [27]


In [27]:
unuseable_fault_index = []
for index in root_index:
    diff_blocks = differences[index]
    num_to_check = [36,37,38]
    if all(num not in diff_blocks for num in num_to_check): # root错了，认证路径没错
        print(f"index: {index:04d}\t data: {diff_blocks}")
        unuseable_fault_index.append(index)

print(len(unuseable_fault_index))

index: 0808	 data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
index: 0809	 data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 27, 28, 29, 30, 32, 34, 35]
index: 0812	 data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
index: 0813	 data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
index: 0816	 data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
index: 0817	 data: [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 34, 35]
index: 0820	 data: [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 3